# nb52 - Cheap methodology round: model soup and EMA weight averaging (Act 4)

**Error analysis.** All architecture/supervision levers are measured (H2-H14); the record stands at 0.0419 with ~0.019 to the metric floor. The verified-evidence sweep ranked two near-free training-level levers we have never applied: weight averaging (EMA/SWA) and cross-seed model soups.

**Question.** Do averaged weights - across training steps (EMA) or across the five nb44 seeds (soup) - improve sigma_eff at zero or near-zero cost?

**Hypotheses.** H16a soup: averaging the five nb44 state_dicts produces a usable model (expected to FAIL - independent seeds are not linearly connected; the test is one eval, so falsify it properly). H16b EMA: an exponential moving average of weights (decay 0.999) beats the best-val checkpoint (published: consistent small gains, arXiv:2502.06761 ICML 2025; tabular 2604.15297).

**Proof criterion.** Config identical to nb44 quantaux; 2 seeds; anchors nb44 singles 0.0442 +/- 0.0005. Win = >0.002; smaller = noise. C-Mixup (2210.05775) is deferred: mixing variable-length masked token sets needs a design pass first - recorded, not skipped silently.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import resolution, PITCH, EPS
from picocal_data import build_grid, prep
from picocal_models import SubNetFQ, QUANTILES, pinball_loss, width_binned_calibration, CFG
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB52_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB52_MODE', 'full')
MBF = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
if MODE == 'smoke': MBF, CLF = MBF[:8], CLF[:4]
t0 = time.time()
ME = build_grid(MBF, 'minbias')
CE = build_grid(CLF, 'clean')
D = prep(4, ME, CE, ng=6)
T = dict(X=torch.from_numpy(D['X']).to(DEVICE), M=torch.from_numpy(D['M']).to(DEVICE),
         G=torch.from_numpy(D['G']).to(DEVICE), Y=torch.from_numpy(D['y']).unsqueeze(1).to(DEVICE),
         E=torch.from_numpy(D['Eraw']).to(DEVICE))
ktr, kva, kte, ctr = D['ktr'], D['kva'], D['kte'], D['ctr']
y = D['y']; Et = D['Et']
QS = torch.tensor(QUANTILES, device=DEVICE)
def batches(idx, bs, rng=None):
    idx = np.asarray(idx)
    if rng is not None: idx = rng.permutation(idx)
    for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
def run_model(model, idx):
    model.eval(); out = []
    with torch.no_grad():
        for b in batches(idx, 256): out.append(model(T['X'][b], T['M'][b], T['G'][b], T['E'][b]).cpu().numpy())
    return np.concatenate(out)
print(f'device {DEVICE} | mode {MODE} | build+prep {time.time()-t0:.0f}s')

minbias: 72554 events


clean: 30303 events


W=4: N 102857 (main 72554 + aux 30303), tr/va/te 50787/10883/10884, IN_DIM 16
device cuda | mode full | build+prep 136s


## H16a - model soup across the five nb44 seeds (one eval, expected to fail)

In [2]:
states = []
for s in range(5):
    ck = CKPT / f'nb44_quantaux_s{s}.pt'
    if ck.exists(): states.append(torch.load(ck, map_location=DEVICE)['bstate'])
print(f'loaded {len(states)} nb44 states')
if len(states) >= 2 and MODE == 'full':
    avg = {k: torch.stack([st[k].float() for st in states]).mean(0) for k in states[0]}
    soup = SubNetFQ(D['IN_DIM'], D['la0'], D['lb0'], ng=6).to(DEVICE)
    soup.load_state_dict(avg)
    pe = width_binned_calibration(run_model(soup, kva), run_model(soup, kte), y[kva])
    print(f'soup ({len(states)} seeds averaged): sigma_eff {resolution(pe, Et[kte])["sigma_eff"]:.4f}  [nb44 singles 0.0442 +/- 0.0005, ens 0.0425]')
else:
    print('skipped (need full mode + nb44 checkpoints)')

loaded 5 nb44 states


soup (5 seeds averaged): sigma_eff 0.2610  [nb44 singles 0.0442 +/- 0.0005, ens 0.0425]


## H16b - EMA of weights during training (decay 0.999)

Same recipe as nb44; the EMA shadow is evaluated for early stopping and the final model. One change only.

In [3]:
def train_ema(seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubNetFQ(D['IN_DIM'], D['la0'], D['lb0'], ng=6).to(DEVICE)
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tr_idx = np.concatenate([np.asarray(ktr), ctr])
    ck = CKPT / f'nb52_ema_s{seed}.pt'
    def vloss(m):
        m.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(kva, 256):
                q = m(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
                d = T['Y'][b] - q
                s += torch.maximum(QS * d, (QS - 1) * d).mean().item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); ema.load_state_dict(st['ema'])
        opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume s{seed} from epoch {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(tr_idx, CFG['batch'], rng):
            opt.zero_grad()
            q = model(T['X'][b], T['M'][b], T['G'][b], T['E'][b])
            d = T['Y'][b] - q
            torch.maximum(QS * d, (QS - 1) * d).mean().backward()
            opt.step()
            ema.update_parameters(model)
        sched.step()
        vv = vloss(ema.module)
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(ema.module.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), ema=ema.state_dict(), opt=opt.state_dict(),
                        sched=sched.state_dict(), best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    final = SubNetFQ(D['IN_DIM'], D['la0'], D['lb0'], ng=6).to(DEVICE)
    final.load_state_dict(bstate)
    pe = width_binned_calibration(run_model(final, kva), run_model(final, kte), y[kva])
    return float(resolution(pe, Et[kte])['sigma_eff']), pe
EPOCHS = {'smoke': 2, 'full': 100}[MODE]
PATIENCE = {'smoke': 99, 'full': 15}[MODE]
SEEDS = {'smoke': [0], 'full': [0, 1, 2, 3, 4]}[MODE]
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb52_ema{TAG}.csv'
done = set()
if CSVP.exists():
    done = set(pd.read_csv(CSVP)['seed'])
    print('resume, done:', sorted(done))
for seed in SEEDS:
    if seed in done: print('skip', seed); continue
    t1 = time.time()
    sig, pe = train_ema(seed, EPOCHS, PATIENCE)
    np.save(OUT / f'nb52_pred{TAG}_ema_s{seed}.npy', pe)
    row = dict(seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t1))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'ema seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
print(pd.read_csv(CSVP).to_string(index=False))

resume, done: [0, 1, 2, 3, 4]
skip 0
skip 1
skip 2
skip 3
skip 4
 seed  sigma_eff  elapsed
    0     0.0425      850
    1     0.0423     1039
    2     0.0428      753
    3     0.0429     1109
    4     0.0421     1338


## Verdict

Anchors: nb44 singles 0.0442 +/- 0.0005 (identical recipe, best-val checkpoint). Win = EMA singles mean better by >0.002; soup evaluated once, reported as-is.

In [4]:
te_e = Et[kte]
edges = np.quantile(te_e, np.linspace(0, 1, 7))
def perbin(pe):
    out = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        out.append(resolution(pe[mm], te_e[mm])['sigma_eff'])
    return out
preds = [np.load(OUT / f'nb52_pred{TAG}_ema_s{s}.npy') for s in SEEDS
         if (OUT / f'nb52_pred{TAG}_ema_s{s}.npy').exists()]
if preds:
    sig = [resolution(p, te_e)['sigma_eff'] for p in preds]
    ens = np.stack(preds).mean(0)
    print(f'ema mean {np.mean(sig):.4f} +/- {np.std(sig):.4f} | ens {resolution(ens, te_e)["sigma_eff"]:.4f}  [nb44 singles 0.0442 +/- 0.0005]')
    print('per-bin ' + ' / '.join(f'{b:.4f}' for b in perbin(ens)))

ema mean 0.0425 +/- 0.0003 | ens 0.0416  [nb44 singles 0.0442 +/- 0.0005]
per-bin 0.0639 / 0.0463 / 0.0355 / 0.0357 / 0.0343 / 0.0352


## Full stack: EMA x5 + D4 TTA + joint sigma_eff-direct calibration

In [5]:
from scipy.optimize import minimize
DI, DJ = 3, 4
meanT = torch.tensor(D['mean'], device=DEVICE); stdT = torch.tensor(D['std'], device=DEVICE)
def tta_views(xb, mb):
    di = xb[:, :, DI] * stdT[DI] + meanT[DI]; dj = xb[:, :, DJ] * stdT[DJ] + meanT[DJ]
    for swap in (False, True):
        for s1 in (1.0, -1.0):
            for s2 in (1.0, -1.0):
                a = (dj if swap else di) * s1; b2 = (di if swap else dj) * s2
                xv = xb.clone()
                xv[:, :, DI] = torch.where(mb, (a - meanT[DI]) / stdT[DI], torch.zeros_like(a))
                xv[:, :, DJ] = torch.where(mb, (b2 - meanT[DJ]) / stdT[DJ], torch.zeros_like(b2))
                yield xv
EMAS = []
for s in range(5):
    ck = CKPT / f'nb52_ema_s{s}.pt'
    if not ck.exists(): continue
    m = SubNetFQ(D['IN_DIM'], D['la0'], D['lb0'], ng=6).to(DEVICE)
    m.load_state_dict(torch.load(ck, map_location=DEVICE)['bstate']); m.eval()
    EMAS.append(m)
print('loaded', len(EMAS), 'EMA models')
def infer(idx, tta):
    per = []
    with torch.no_grad():
        for mdl in EMAS:
            out = []
            for j in range(0, len(idx), 256):
                b = torch.from_numpy(np.asarray(idx[j:j+256])).to(DEVICE)
                xb, mb2 = T['X'][b], T['M'][b]
                if tta:
                    qs = [mdl(xv, mb2, T['G'][b], T['E'][b]) for xv in tta_views(xb, mb2)]
                    out.append(torch.stack(qs).mean(0).cpu().numpy())
                else:
                    out.append(mdl(xb, mb2, T['G'][b], T['E'][b]).cpu().numpy())
            per.append(np.concatenate(out))
    return np.stack(per)
qv = infer(kva, True).mean(0); qt = infer(kte, True).mean(0)
yva = y[kva]; Ev = np.exp(yva)
wv = qv[:, 2] - qv[:, 0]; wt_ = qt[:, 2] - qt[:, 0]
cuts = np.quantile(wv, [1/3, 2/3])
gv = np.digitize(wv, cuts); gt = np.digitize(wt_, cuts)
p0 = []
for g in range(3):
    a0, b0 = np.polyfit(qv[gv == g, 1], yva[gv == g], 1)
    p0 += [a0, b0]
def apply(p, q, grp):
    pe = np.empty(len(q))
    for g in range(3):
        pe[grp == g] = np.exp(p[2*g] * q[grp == g, 1] + p[2*g+1])
    return pe
def obj(p): return resolution(apply(p, qv, gv), Ev)['sigma_eff']
res = minimize(obj, p0, method='Nelder-Mead', options=dict(xatol=1e-5, fatol=1e-7, maxiter=3000))
p = res.x if res.fun <= obj(np.array(p0)) else np.array(p0)
pe_ls = apply(np.array(p0), qt, gt)
pe_stack = apply(p, qt, gt)
np.save(OUT / 'nb52_pred_stack.npy', pe_stack)
te_e2 = Et[kte]
print(f'EMAx5 + TTA (LS calib):     {resolution(pe_ls, te_e2)["sigma_eff"]:.4f}')
print(f'EMAx5 + TTA + direct calib: {resolution(pe_stack, te_e2)["sigma_eff"]:.4f}   [prev record 0.0419 | targets 0.06/0.045/0.035/0.032/0.030/0.030]')
edges2 = np.quantile(te_e2, np.linspace(0, 1, 7))
bins = []
for i in range(6):
    hi = edges2[i+1] + (1e-9 if i == 5 else 0)
    mm = (te_e2 >= edges2[i]) & (te_e2 < hi)
    bins.append(f'{resolution(pe_stack[mm], te_e2[mm])["sigma_eff"]:.4f}')
print('per-bin ' + ' / '.join(bins))

loaded 5 EMA models


EMAx5 + TTA (LS calib):     0.0415
EMAx5 + TTA + direct calib: 0.0409   [prev record 0.0419 | targets 0.06/0.045/0.035/0.032/0.030/0.030]
per-bin 0.0629 / 0.0465 / 0.0344 / 0.0350 / 0.0342 / 0.0342
